In [ ]:
!pip install azure-search-documents
!pip install langchain langchain-openai langchain-experimental
!pip install sentence-transformers>=2.7.0 transformers>=4.51.0
!pip install pyyaml tqdm

In [ ]:
"""

====================
Pipeline de chunking semántico + generación de embeddings para el corpus
documental de AMC Global.

Flujo:
    1. Lee configuración desde config_chunking.yaml.
    2. Recorre todos los .json del corpus_dir.
    3. Extrae 'texto_completo' de cada documento.
    4. Aplica Semantic Chunking con text-embedding-3-small (OpenAI).
    5. Genera embeddings finales de los chunks con Qwen3-Embedding-0.6B (local).
    6. Construye objetos chunk con metadatos y un hash SHA-256 como PK.
    7. Guarda todos los chunks en chunks_output.json.

"""

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import json
import hashlib
from pathlib import Path
import re

import yaml
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from huggingface_hub import login

#Importaciones de los servicios de Azure
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
)


In [ ]:
# ── Carga de configuración ───────────────────────────────────────────────────

def cargar_config(ruta) -> dict:
    with open(ruta, encoding="utf-8") as f:
        return yaml.safe_load(f)


## Chunking
---
Se empleó chunking semántico como estrategia de segmentación del corpus documental. A diferencia del chunking por tamaño fijo, que divide el texto en fragmentos de longitud predeterminada independientemente de su contenido, el chunking semántico detecta los puntos de ruptura temática calculando la similitud entre frases consecutivas en el espacio de embeddings. Cuando la similitud entre dos frases cae por debajo de un umbral estadístico, el sistema infiere que se ha producido un cambio de tema e introduce un corte. [Decir aquí qué tipo de umbral estadístico estamos usando nosotros]


Este enfoque produce chunks semánticamente coherentes, lo que mejora la calidad de la recuperación al garantizar que cada fragmento indexado representa una unidad de significado completa y no un extracto arbitrario del texto original.

Para llevar a cabo el chunking semántico, el modelo de embedding utilizado en la comparación de similitudes entre secuencias del texto es `text-embedding-3-small` de OpenAI. En esta fase del proceso, el modelo de embedding actúa exclusivamente como detector de cambios semánticos entre frases consecutivas, sin que sus vectores sean almacenados ni utilizados en el sistema de recuperación. Por tanto, no se requiere el mismo nivel de precisión que sí necesitará el retrieval, lo que justifica el uso de un modelo ligero y de bajo coste para esta tarea.

Sin embargo, para la generación de los embeddings finales de los chunks, que son los que se almacenarán en la base de datos vectorial, se empleó el modelo `Qwen3-Embedding-0.6B` (Zhang et al., 2025). La elección de este modelo frente a alternativas comerciales como `text-embedding-3-large` de OpenAI se fundamenta en los resultados reportados en su evaluación sobre los marcos de referencia MTEB (Muennighoff et al., 2023) y MMTEB (Enevoldsen et al., 2025), que constituyen el estándar de facto para la comparación objetiva de modelos de embedding en la comunidad científica. A pesar de su reducido tamaño de 0.6 mil millones de parámetros, el modelo Qwen3-Embedding-0.6B supera sistemáticamente al modelo `text-embedding-3-large` de OpenAI en múltiples dominios de evaluación: en el benchmark MMTEB multilingüe obtiene una puntuación media de 64.33 frente al 58.93 del modelo de OpenAI; en MTEB inglés (v2) alcanza un 70.70 frente al 66.43; y en MTEB de código logra un 75.41 frente al 58.95. Este rendimiento superior es atribuido por Zhang et al. (2025) a su proceso de entrenamiento multietapa, que combina un preentrenamiento débilmente supervisado sobre datos sintéticos generados por el propio modelo base Qwen3, un ajuste fino supervisado con datos de alta calidad, y técnicas de fusión de modelos, lo que permite al Qwen3-Embedding-0.6B superar a modelos propietarios de mayor tamaño en entornos multilingües. Dado que el corpus documental de AMC Global está íntegramente en español, el rendimiento superior de este modelo en el benchmark multilingüe resulta especialmente relevante para garantizar la calidad del retrieval en el sistema RAG desarrollado.

Se usó el mismo modelo tanto para la generación de los embeddings de los chunks como para la representación de las consultas en tiempo de inferencia, esto responde a una necesidad metodológica fundamental. En un sistema de recuperación densa, la búsqueda por similitud opera comparando el vector de la consulta del usuario con los vectores de los chunks almacenados en la base de datos vectorial. Esta comparación es matemáticamente válida únicamente si ambos vectores han sido generados por el mismo modelo de embedding, ya que cada modelo tiene su propio espacio vectorial con su propia geometría y distribución semántica. Si se utilizaran modelos distintos para indexar los chunks y para codificar las consultas, los vectores resultantes pertenecerían a espacios incompatibles lo, que haría que las métricas de similitud, como la similitud coseno, carecieran de validez semántica y produjeran resultados de recuperación incorrectos. Por tanto, el modelo `Qwen3-Embedding-0.6B` se empleó de forma consistente en ambas etapas del pipeline RAG: durante la indexación del corpus y durante la inferencia en producción.

---

### Referencias

Enevoldsen, K., Chung, I., Kerboua, I., Kardos, M., Mathur, A., Stap, D., Gala, J., et al. (2025). *MMTEB: Massive multilingual text embedding benchmark*. arXiv:2502.13595. https://arxiv.org/abs/2502.13595

Muennighoff, N., Tazi, N., Magne, L., & Reimers, N. (2023). MTEB: Massive text embedding benchmark. *Proceedings of the 17th Conference of the European Chapter of the Association for Computational Linguistics*, 2014–2037. https://arxiv.org/abs/2210.07316

Zhang, Y., Li, M., Long, D., Zhang, X., Lin, H., Yang, B., Xie, P., Yang, A., Liu, D., Lin, J., Huang, F., & Zhou, J. (2025). *Qwen3 Embedding: Advancing text embedding and reranking through foundation models*. arXiv:2506.05176. https://arxiv.org/abs/2506.05176

Desde el punto de vista arquitectónico, conviene distinguir sus características. La **dimensión del embedding** hace referencia al tamaño del vector de representación que el modelo genera para cada fragmento de texto; una mayor dimensionalidad implica un espacio vectorial más rico, aunque no necesariamente una mayor calidad de retrieval, como demuestran los benchmarks MTEB y MMTEB discutidos anteriormente. La **longitud de contexto máxima**, por su parte, determina el número máximo de tokens que el modelo puede procesar en una única entrada sin truncar el texto. El modelo `Qwen3-Embedding-0.6B` produce vectores más compactos de 1024 dimensiones frente a las 3072 del modelo `text-embedding-3-large`, lo que reduce el coste de almacenamiento en la base de datos vectorial y acelera las operaciones de búsqueda por similitud. Sin embargo, su ventana de contexto de entrada es notablemente superior, admitiendo hasta 32.768 tokens frente a los 8.191 del modelo de OpenAI. Esta característica resulta especialmente relevante para el corpus documental de AMC Global, que incluye documentos de elevada extensión como normativas legales, cuya segmentación podría generar chunks de considerable longitud.

| Característica | `Qwen3-Embedding-0.6B` | `text-embedding-3-large` |
|---|---|---|
| Dimensión del embedding | 1.024 | 3.072 |
| Contexto máximo de entrada (tokens) | 32.768 | 8.191 |

In [ ]:

def limpiar_texto_completo(texto: str) -> str:
    # 1. Eliminar líneas que son solo un número de sección
    texto = re.sub(r'^\s*\d+(\.\d+)*\.?\s*$', '', texto, flags=re.MULTILINE)
    # 2. Eliminar líneas de índice con puntos de relleno (TÍTULO.......N)
    texto = re.sub(r'^.+\.{4,}\s*\d+\s*$', '', texto, flags=re.MULTILINE)
    # 3. Eliminar líneas de puntos dispersos (. . . . . .)
    texto = re.sub(r'^[\s.]{3,}$', '', texto, flags=re.MULTILINE)
    # 4. Colapsar saltos de línea múltiples
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()



In [ ]:
# ── Funciones auxiliares ─────────────────────────────────────────────────────
def es_chunk_valido(texto: str) -> bool:
    t = texto.strip()
    # Eliminar si solo contiene puntos y espacios
    if re.fullmatch(r'[\s.]+', t):
        return False
    # Eliminar si es solo un número con punto opcional
    if re.fullmatch(r'\d+\.?', t):
        return False
    # Eliminar referencias de página
    if re.fullmatch(r'Página\s*\d+.*', t, re.IGNORECASE):
        return False
    # Eliminar si tiene menos de 4 palabras Y no contiene letras suficientes
    palabras = t.split()
    if len(palabras) <= 2 and len(t) < 10:
        return False
    return True


def generar_id(texto: str) -> str:
    """Hash SHA-256 del texto como identificador único del chunk (PK en BD)."""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def cargar_documento(ruta: Path) -> dict:
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)


def procesar_documento(ruta: Path, chunker: SemanticChunker, embedding_model: SentenceTransformer) -> list[dict]:
    """
    Procesa un documento JSON:
        1. Extrae 'texto_completo'.
        2. Aplica Semantic Chunking.
        3. Genera embeddings en bloque con Qwen3-Embedding-0.6B.
        4. Construye objetos chunk con metadatos.
    """
    doc   = cargar_documento(ruta)
    texto = doc.get("texto_completo", "").strip()

    texto = limpiar_texto_completo(texto)
    if not texto:
        print(f"  [AVISO] {ruta.name}: 'texto_completo' vacío. Omitido.")
        return []

    nombre_documento = doc.get("nombre_procesado")
    nombre_original  = doc.get("nombre_original")
    num_paginas      = doc.get("num_paginas", None)
    fecha_procesado  = doc.get("fecha_procesado", "")

    # 1. Chunking semántico; Simplemente se usa el wrapper del chunker proporcionado por
    # LangChain para hacer el chunking conforme a la estrategia escogida.
    chunks_texto = chunker.split_text(texto)
    chunks_texto = [c for c in chunks_texto if es_chunk_valido(c)]
    if not chunks_texto:
        chunks_texto = [texto]  # fallback para documentos muy cortos

    # 2. Embeddings en bloque: una sola pasada por el modelo para todo el documento
    # No se usa prompt_name porque estos son documentos, no queries
    embeddings_lista = embedding_model.encode(
        chunks_texto,
        batch_size=4,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).tolist()

    # Liberar caché de GPU tras cada documento
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 3. Construcción de objetos chunk con metadatos
    total     = len(chunks_texto)
    resultado = []

    for idx, (texto_chunk, vector) in enumerate(zip(chunks_texto, embeddings_lista)):
        resultado.append({
            "id":        generar_id(texto_chunk),
            "texto":     texto_chunk,
            "embedding": vector,
            "metadata": {
                "nombre_documento": nombre_documento,
                "nombre_original":  nombre_original,
                "num_paginas":      num_paginas,
                "fecha_procesado":  fecha_procesado,
                "chunking":         "semantic_chunking",
                "chunk_index":      idx,
                "total_chunks_doc": total,
            }
        })

    return resultado


MAIN


In [ ]:
config = cargar_config("...")

In [ ]:


# ── Modelo para el SemanticChunker (OpenAI) ──────────────────────────────────
# Usado exclusivamente para detectar cortes semánticos entre frases.
# Sus vectores no se almacenan ni se usan en el RAG.

openai_apitoken = config.get("api_keys", {}).get("openai")

embedding_chunker = OpenAIEmbeddings(
    model = config.get("models").get("semantic_chunker").get("name", "text-embedding-3-small"),
    api_key = openai_apitoken
)

chunker = SemanticChunker(
    embeddings=embedding_chunker,
    breakpoint_threshold_type=config.get("chunking").get("breakpoint_type"),
    breakpoint_threshold_amount=config.get("chunking").get("breakpoint_amount"),
)

# ── Modelo para embeddings finales (Qwen3-Embedding-0.6B local) ──────────────
# Usado para generar los vectores que se guardarán en la BD vectorial
# y que también se usarán para codificar las queries en producción.
# Ambos usos comparten modelo para garantizar compatibilidad en el espacio vectorial.

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cargando Qwen3-Embedding-0.6B en {device}...")

login(token=config.get("models").get("embeddings").get("hf_token"))

embedding_model = SentenceTransformer(
    config.get("models").get("embeddings").get("name", "Qwen/Qwen3-Embedding-0.6B"),
    device=device,
)

corpus_path = Path(config.get("paths").get("corpus_dir"))
output_file = config.get("paths").get("output_file")
archivos    = sorted(corpus_path.glob("*.json"))


Cómo funciona el percentil en el SemanticChunker
--
El chunker procesa el texto frase a frase y calcula la similitud de embedding entre cada par de frases consecutivas. Imagina que tienes un documento con 100 frases: obtienes 99 valores de similitud, uno por cada par adyacente.
Esos 99 valores forman una distribución. Algunos pares de frases son muy similares (similitud alta, misma idea) y otros son muy distintos (similitud baja, cambio de tema).
El percentil 85 significa: "encuentra el valor de similitud que deja por debajo al 85% de todos los pares, y úsalo como umbral de corte". Cualquier par de frases cuya similitud caiga por debajo de ese umbral se interpreta como un salto semántico y ahí se introduce un corte.
Dicho de forma simple: el 85 significa que solo el 15% de los pares de frases más disimilares generarán un corte. Si bajas a 70, el 30% generará cortes y tendrás chunks más pequeños. Si subes a 95, solo el 5% generará cortes y tendrás chunks más grandes.

In [ ]:

if not archivos:
    print(f"No se encontraron archivos .json en '{corpus_path}'.")
    raise Exception("No se encontraron archivos .json en el corpus.")

print(f"Documentos encontrados  : {len(archivos)}")
print(f"Chunker                 : {config.get('chunking').get('breakpoint_type')} @ {config.get('chunking').get('breakpoint_amount')}")
print(f"Modelo chunker          : {config.get('models').get('semantic_chunker').get('name')}")
print(f"Modelo embeddings       : {config.get('models').get('embeddings').get('name')}")
print(f"Dispositivo             : {device}\n")

todos_los_chunks = []

for ruta in tqdm(archivos, desc="Procesando documentos"):
    chunks = procesar_documento(ruta, chunker, embedding_model)
    todos_los_chunks.extend(chunks)
    tqdm.write(f"  {ruta.name}: {len(chunks)} chunks")

# Crear directorio de salida si no existe
Path(output_file).parent.mkdir(parents=True, exist_ok=True)


with open(output_file, "w", encoding="utf-8") as f:
    json.dump(todos_los_chunks, f, ensure_ascii=False, indent=2)

print(f"\n{'─' * 52}")
print(f"  Documentos procesados  : {len(archivos)}")
print(f"  Total chunks generados : {len(todos_los_chunks)}")
print(f"  Archivo de salida      : {output_file}")
print(f"{'─' * 52}")
print("\nSiguiente paso: indexar chunks_output.json en Azure AI Search.")




Filtrado de chunks no válidos
--
Tras aplicar el chunking semántico, se detectó la presencia de fragmentos sin valor semántico generados como artefacto del proceso de extracción de texto desde PDF. Estos fragmentos incluían líneas compuestas únicamente por puntos dispersos, números de sección huérfanos, referencias de página aisladas y bloques derogados sin contenido informativo. Su presencia en el índice vectorial introduciría ruido en el retrieval, ya que podrían ser recuperados ante consultas con las que comparten similitud superficial sin aportar información útil al LLM. Por este motivo, se aplicó un filtro post-chunking que descarta aquellos fragmentos que, por su forma, no pueden contener información semántica relevante, preservando en todo caso los títulos de artículos y epígrafes breves que, aunque cortos, constituyen unidades de significado válidas dentro del corpus.
Se empleó chunking semántico como estrategia de segmentación del corpus documental. A diferencia del chunking por tamaño fijo, que divide el texto en fragmentos de longitud predeterminada independientemente de su contenido, el chunking semántico detecta los puntos de ruptura temática calculando la similitud entre frases consecutivas en el espacio de embeddings. Cuando la similitud entre dos frases cae por debajo de un umbral estadístico, el sistema infiere que se ha producido un cambio de tema e introduce un corte. Este enfoque produce chunks semánticamente coherentes, lo que mejora la calidad del retrieval al garantizar que cada fragmento indexado representa una unidad de significado completa y no un extracto arbitrario del texto original.

In [ ]:
for c in todos_los_chunks:
    texto = c["texto"].strip()
    if len(texto) < 25:
        print(f"[{c['metadata']['nombre_documento']}] '{texto}'")

## Creación del Índice en la BD de Azure AI Search.
---

In [ ]:
config = cargar_config("...")

In [ ]:
"""
==========================
Pipeline de creación del índice e indexación de chunks en Azure AI Search.

Flujo:
    1. Lee configuración desde config_chunking.yaml.
    2. Crea el índice 'idx-tfg-jaac' en Azure AI Search con esquema híbrido
       (campos de texto + campo vectorial de 1024 dimensiones).
    3. Lee chunks_output.json y sube los chunks en lotes de 100.


"""




ENDPOINT  = config["api_keys"]["azure_ai_search"]["endpoint"]
API_KEY   = config["api_keys"]["azure_ai_search"]["api_key"]
INDEX_NAME = "idx-tfg-jaac"

# Dimensión del embedding de Qwen3-Embedding-0.6B
EMBEDDING_DIM = 1024

# Tamaño del lote de subida — Azure AI Search acepta máximo 1000 docs por lote
# Usamos 100 para no saturar la memoria ni la API
BATCH_SIZE = 100

credential = AzureKeyCredential(API_KEY)

# ── Paso 1: Crear el índice ──────────────────────────────────────────────────

def crear_indice():
    """
    Define el esquema del índice y lo crea en Azure AI Search.
    Si el índice ya existe, lo elimina y lo vuelve a crear.

    Campos del esquema:
        - id:                 PK (hash SHA-256 del texto del chunk)
        - texto:              contenido textual del chunk (búsqueda BM25)
        - embedding:          vector de 1024 dims (búsqueda vectorial HNSW)
        - nombre_documento:   metadato filtrable
        - nombre_original:    metadato filtrable
        - num_paginas:        metadato
        - fecha_procesado:    metadato
        - chunking:           estrategia usada (semantic_chunking)
        - chunk_index:        posición del chunk en el documento
        - total_chunks_doc:   total de chunks del documento
    """
    index_client = SearchIndexClient(ENDPOINT, credential)

    # Eliminar índice si ya existe (útil para re-indexaciones)
    existing = [idx.name for idx in index_client.list_indexes()]
    if INDEX_NAME in existing:
        index_client.delete_index(INDEX_NAME)
        print(f"Índice '{INDEX_NAME}' eliminado para recrearlo.")

    # Configuración del algoritmo HNSW para búsqueda vectorial
    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="hnsw-config",
                parameters={
                    "m": 4,                 # conexiones por nodo en el grafo
                    "efConstruction": 400,  # precisión en construcción
                    "efSearch": 500,        # precisión en búsqueda
                    "metric": "cosine",     # similitud coseno
                }
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="vector-profile",
                algorithm_configuration_name="hnsw-config",
            )
        ]
    )

    # Definición de campos del índice
    fields = [
        SimpleField(
            name="id",
            type=SearchFieldDataType.String,
            key=True,                    # PK del índice
            filterable=True,
        ),
        SearchableField(
            name="texto",
            type=SearchFieldDataType.String,
            analyzer_name="es.microsoft", # analizador en español para BM25
        ),
        SearchField(
            name="embedding",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=EMBEDDING_DIM,
            vector_search_profile_name="vector-profile",
        ),
        SimpleField(name="nombre_documento", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SimpleField(name="nombre_original",  type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="num_paginas",      type=SearchFieldDataType.Int32,  filterable=True),
        SimpleField(name="fecha_procesado",  type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="chunking",         type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="chunk_index",      type=SearchFieldDataType.Int32,  filterable=True),
        SimpleField(name="total_chunks_doc", type=SearchFieldDataType.Int32,  filterable=True),
    ]

    index = SearchIndex(
        name=INDEX_NAME,
        fields=fields,
        vector_search=vector_search,
    )

    index_client.create_index(index)
    print(f"Índice '{INDEX_NAME}' creado correctamente.")
    print(f"  Campos: {len(fields)}")
    print(f"  Dimensión embedding: {EMBEDDING_DIM}")
    print(f"  Algoritmo vectorial: HNSW (cosine)")


In [ ]:


# ── Paso 2: Indexar los chunks ───────────────────────────────────────────────

def indexar_chunks():
    """
    Lee chunks_output.json y sube todos los chunks al índice en lotes.
    Cada chunk se transforma para que sus metadatos queden en el nivel
    raíz del documento, que es el formato que espera Azure AI Search.
    """
    output_file = config["paths"]["output_file"]

    print(f"\nLeyendo chunks desde: {output_file}")
    with open(output_file, encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"Total chunks a indexar: {len(chunks)}")

    search_client = SearchClient(ENDPOINT, INDEX_NAME, credential)

    # Transformar cada chunk al formato plano que espera Azure AI Search
    # Los metadatos pasan del campo 'metadata' al nivel raíz del documento
    def transformar(chunk: dict) -> dict:
        meta = chunk.get("metadata", {})
        return {
            "id":                chunk["id"],
            "texto":             chunk["texto"],
            "embedding":         chunk["embedding"],
            "nombre_documento":  meta.get("nombre_documento"),
            "nombre_original":   meta.get("nombre_original"),
            "num_paginas":       meta.get("num_paginas"),
            "fecha_procesado":   meta.get("fecha_procesado"),
            "chunking":          meta.get("chunking"),
            "chunk_index":       meta.get("chunk_index"),
            "total_chunks_doc":  meta.get("total_chunks_doc"),
        }

    documentos = [transformar(c) for c in chunks]

    # Subida en lotes
    errores = 0
    for i in tqdm(range(0, len(documentos), BATCH_SIZE), desc="Subiendo lotes"):
        lote = documentos[i:i + BATCH_SIZE]
        resultado = search_client.upload_documents(documents=lote)
        errores += sum(1 for r in resultado if not r.succeeded)

    print(f"\n{'─' * 50}")
    print(f"  Chunks subidos    : {len(documentos) - errores}")
    print(f"  Errores           : {errores}")
    print(f"  Índice            : {INDEX_NAME}")
    print(f"  Endpoint          : {ENDPOINT}")
    print(f"{'─' * 50}")


El índice vectorial de Azure AI Search se construyó empleando el algoritmo HNSW con similitud coseno como métrica de comparación entre vectores, configurado con los parámetros m=4, efConstruction=400 y efSearch=500. El parámetro m define el número de conexiones bidireccionales que cada nodo establece con sus vecinos durante la construcción del grafo; un valor reducido es adecuado para corpus de pequeña escala como el empleado en este trabajo, donde la densidad de conexiones necesaria para una navegación eficiente es baja. El parámetro efConstruction controla el número de candidatos examinados al insertar cada nodo en el grafo durante la indexación, determinando la precisión de la estructura resultante a costa de un mayor tiempo de construcción. El parámetro efSearch regula el número de candidatos evaluados en cada consulta, estableciendo el equilibrio entre precisión de recuperación y latencia en tiempo de inferencia. La similitud coseno fue seleccionada como métrica de comparación por ser el estándar empleado por los modelos de embedding basados en transformers, donde los vectores son normalizados durante el entrenamiento y la similitud coseno equivale al producto escalar entre vectores unitarios.

In [ ]:


# ── Ejecución ────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    crear_indice()
    indexar_chunks()

## Pruebas correcta recuperación de la BD

In [ ]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

search_client = SearchClient(ENDPOINT, INDEX_NAME, credential)

# 1. Contar documentos
count = search_client.get_document_count()
print(f"Documentos en el índice: {count}")

# 2. Buscar un chunk concreto por texto
resultados = search_client.search(
    search_text="código ético AMC",
    top=3,
    select=["id", "texto", "nombre_documento"]
)

print("\nPrueba de búsqueda léxica:")
for r in resultados:
    print(f"\n  Documento: {r['nombre_documento']}")
    print(f"  Texto: {r['texto'][:150]}")

In [ ]:
from sentence_transformers import SentenceTransformer

# Cargar el modelo de embeddings (ya debería estar en caché)
embedding_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cpu")

# Generar embedding de una query de prueba
query = "¿Cuáles son las pautas de conducta ética de AMC?"
query_vector = embedding_model.encode(query, convert_to_numpy=True).tolist()

# Búsqueda vectorial pura
resultados_vector = search_client.search(
    search_text=None,
    vector_queries=[{
        "kind": "vector",
        "vector": query_vector,
        "fields": "embedding",
        "k": 3
    }],
    select=["id", "texto", "nombre_documento"]
)

print("Prueba de búsqueda vectorial:")
for r in resultados_vector:
    print(f"\n  Documento: {r['nombre_documento']}")
    print(f"  Score: {r['@search.score']:.4f}")
    print(f"  Texto: {r['texto'][:150]}")

In [ ]:
# Detectar IDs duplicados en el JSON de chunks
ids = [c["id"] for c in chunks]
duplicados = len(ids) - len(set(ids))
print(f"IDs duplicados en chunks_output.json: {duplicados}")

In [ ]:
import json

output_file = config["paths"]["output_file"]

with open(output_file, encoding="utf-8") as f:
    chunks = json.load(f)

ids = [c["id"] for c in chunks]
duplicados = len(ids) - len(set(ids))
print(f"Total chunks en JSON : {len(ids)}")
print(f"IDs duplicados       : {duplicados}")

In [ ]:
from collections import Counter

id_counts = Counter(c["id"] for c in chunks)
duplicados = {id_: count for id_, count in id_counts.items() if count > 1}

for id_, count in duplicados.items():
    chunk = next(c for c in chunks if c["id"] == id_)
    print(f"\nID: {id_[:20]}... (aparece {count} veces)")
    print(f"Documento: {chunk['metadata']['nombre_documento']}")
    print(f"Texto: '{chunk['texto'][:100]}'")